In [1]:
from dataclasses import dataclass

## States and Symbols

Our Finite State Machines needs to have some `State` objects and an Alphabet of `Symbol` objects.

I've use a `dataclass` here because there is not much that we need to store. The setting of `eq` and `frozen` to `True` means that we get _hashable_ objects that I can use as dictionary keys later (because they are immutable).

In [2]:
@dataclass(eq=True, frozen=True)
class State:
    name: str = ''

In [3]:
@dataclass(eq=True, frozen=True)
class Symbol:
    name: str = ''

In [4]:
S0 = State('S0')
S1 = State('S1')
S2 = State('S2')

In [5]:
a = Symbol('a')
b = Symbol('b')

## Finite State Machine

Our Finite State Machine needs:

   - an alphabet of symbols representing the transitions
   - a set of states
   - an initial state

It also has an optional list of _accepted states_.

We also need to be able to add and store state transitions.


In [6]:
class StateMachine:
    def __init__(self, states, alphabet, initial_state, accepted_states=None):
        if accepted_states is None:
            accepted_states = []
        self.states = states
        self.alphabet = alphabet

        self.initial_state = initial_state

        if accepted_states:
            self.accepted_states = accepted_states
        else:
            self.accepted_states = []

        self.state_transitions = {s: {} for s in states}

    def add_transition(self, symbol, from_state, to_state):
        self.state_transitions[from_state][symbol] = to_state

In [7]:
FSM = StateMachine([S0, S1, S2], [a, b], S0, [S1])


In [8]:
FSM.add_transition(a, S0, S1)


In [9]:
FSM.state_transitions

{State(name='S0'): {Symbol(name='a'): State(name='S1')},
 State(name='S1'): {},
 State(name='S2'): {}}

In [10]:

FSM.add_transition(a, S1, S1)
FSM.add_transition(b, S1, S2)
FSM.add_transition(b, S2, S2)
FSM.add_transition(a, S2, S0)
FSM.add_transition(b, S0, S2)

In [11]:
FSM.state_transitions

{State(name='S0'): {Symbol(name='a'): State(name='S1'),
  Symbol(name='b'): State(name='S2')},
 State(name='S1'): {Symbol(name='a'): State(name='S1'),
  Symbol(name='b'): State(name='S2')},
 State(name='S2'): {Symbol(name='b'): State(name='S2'),
  Symbol(name='a'): State(name='S0')}}

## Acceptor

Now that we have our Finite State Machine and we have stored our states, alphabet and transitions, we need to be able to test to see which strings of symbols are _acceptable_ to the FSM.

This is the role of our `Acceptor` class. It simply stores a current state and then performs transitions according to the symbols that it receives. It then records whether the submitted symbol puts the system into an acceptable state - in this case the acceptor is 'accepting' the sequence thus far. Resetting the acceptor returns the system to its default state.

As a bonus the `path` attribute tells us the sequence of states that the sequence of symbols has generated.

There is also a method `Acceptor.accept()` which will run through a given sequence and return whether the system finishes in an acceptable state.

In [12]:
class Acceptor:
    def __init__(self, fsm: 'StateMachine'):
        self.fsm = fsm
        self.path = None
        self.current_state = None
        self.accepting = None
        self.reset()

    def receive(self, symbol: Symbol):
        if symbol in self.fsm.state_transitions[self.current_state]:
            self.current_state = self.fsm.state_transitions[self.current_state][symbol]
            self.path.append(self.current_state)

        if self.current_state in self.fsm.accepted_states:
            self.accepting = True
        else:
            self.accepting = False

        return self.accepting

    def reset(self):
        self.accepting = True
        self.current_state = self.fsm.initial_state
        self.path = [self.current_state]

    def accept(self, sequence: [Symbol]):
        self.reset()
        for s in sequence:
            self.receive(s)

        return self.accepting


In [13]:
acceptor = Acceptor(FSM)

In [14]:
acceptor.accept([b, b, a])

False

In [15]:
acceptor.accept([a, b, b, a, a])

True

In [16]:
acceptor.path


[State(name='S0'),
 State(name='S1'),
 State(name='S2'),
 State(name='S2'),
 State(name='S0'),
 State(name='S1')]

In [17]:
acceptor.reset()
acceptor.receive(b)


False

In [18]:
acceptor.receive(b)


False

In [19]:
acceptor.receive(a)

False